# Bihar – FarMech beneficiary list scraper

Iterates Financial Year → District → Block → Gram Panchayat on the [Bihar FarMech portal](https://farmech.bih.nic.in/FMNEW/BenefGPListFrom1920.aspx), reads each GP's beneficiary table and saves one Excel file per year to `data/bihar/`.

The `l_*` variables below let you resume a long run from the last completed code.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common import exceptions
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.alert import Alert

In [ ]:
from selenium.webdriver.chrome.service import Service

In [ ]:
from selenium.common.exceptions import ElementNotInteractableException
from selenium.common.exceptions import UnexpectedAlertPresentException

In [ ]:
import time

In [ ]:
import os

In [ ]:
import pandas as pd

In [ ]:
url ='https://farmech.bih.nic.in/FMNEW/BenefGPListFrom1920.aspx'

In [ ]:
base_loc=os.path.join(os.getcwd(),'data','bihar')
if not os.path.exists(base_loc):
    os.makedirs(base_loc)

In [ ]:
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1200")

In [ ]:
####### resume boundary #########
# Codes already scraped; set lower to start earlier (use "0" to scrape everything).
l_gp="2043100221"
l_b="2042"
l_d="216"
l_y="0"

In [ ]:
def data_append(df):
    global data_df
    # data_df=data_df.append(df)
    data_df=pd.concat([data_df,df],axis=0)

In [ ]:
def h1_driver(driver):
    try:
        alert = driver.switch_to.alert
        alert.dismiss()
    except:
        pass
    h1 = driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_F_year')
    dist = h1.find_elements(By.TAG_NAME, "option")
    
    dist_val=[]
    for d in dist:
        dist_val.append(d.get_attribute("value"))
    
    for d in dist_val:
        
        district=d
        if d=='Select':
            continue
        if not d=='Select':
            print(district)
            h2_driver(district,driver,d)
            # element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_btn_cancel")))
            driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_cancel").click()
            try:
                alert = driver.switch_to.alert
                alert.dismiss()
            except:
                pass
    return 

In [ ]:
def h2_driver(district,driver,k):
    try:
        alert = driver.switch_to.alert
        alert.dismiss()
    except:
        pass
    
    e1=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_F_year'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_proceed").click()
    h2 = driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_dist')
    all_options = h2.find_elements(By.TAG_NAME, "option")
    print(len(all_options),district)
    
    block_val=[]
    for d in all_options:
        block_val.append(d.get_attribute("value"))
    global l_d
    
    for b in block_val:
        block=b
        if b=='Select':
            continue
        if not b=='Select':
            if (int(block)>int(l_d)) and (int(block)!=611):
                print('h2_driver (s)',block,district)
                h21_driver(block,district,driver,d)
                # element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_btn_cancel")))
                driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_cancel").click()
                print('h2_driver (e)',block,district)
                try:
                    alert = driver.switch_to.alert
                    alert.dismiss()
                except:
                    pass


In [ ]:
def h21_driver(block,district,driver,d):
    try:
        alert = driver.switch_to.alert
        alert.dismiss()
    except:
        pass
    
    e1=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_F_year'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_proceed").click()
    
    h2 = driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_dist')
    s_e2=Select(h2)
    s_e2.select_by_value(block)

    h21=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_block'))
    all_options = h21.find_elements(By.TAG_NAME, "option")
    
    global l_b
    
    gram_val=[]
    for d in all_options:
        gram_val.append(d.get_attribute("value"))

    for b in gram_val:
        gram=b
        if b=='Select':
            continue
        if not b=='Select':
            if (int(gram)>int(l_b)):
                print('h21_driver (s)',gram,block,district)
                h3_driver(gram,block,district,driver,d)
                # element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_btn_cancel")))
                driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_cancel").click()
                print('h21_driver (e)',gram,block,district)
                try:
                    alert = driver.switch_to.alert
                    alert.dismiss()
                except:
                    pass


In [ ]:
def h3_driver(gram,block,district,driver,d):
    try:
        alert = driver.switch_to.alert
        alert.dismiss()
    except:
        pass
    
    e1=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_F_year'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_proceed").click()
    
    h2 = driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_dist')
    s_e2=Select(h2)
    s_e2.select_by_value(block)
    
    h21=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_block'))
    s_e21=Select(h21)
    s_e21.select_by_value(gram)
    
    h3=driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_gpanchayat')
    all_options = h3.find_elements(By.TAG_NAME, "option")
    
    gp_val=[]
    for d in all_options:
        gp_val.append(d.get_attribute("value"))

    global l_gp
    for b in gp_val:
        gp=b
        if b=='Select':
            continue
        if not b=='Select':
            print("h3_driver (s)",b,gram,block,district)
            if (int(gp)>int(l_gp)):
                print("##########",gp,gram,block,district)
                h4_driver(gp,gram,block,district,driver,d)
                # element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_btn_cancel")))
                try:
                    driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_cancel").click()
                    print("h3_driver (e)",b,gram,block,district)
                except UnexpectedAlertPresentException:
                    print('Except loop')
                    try:
                        alert = driver.switch_to.alert
                        alert.dismiss()
                    except:
                        pass

In [ ]:
def h4_driver(gp,gram,block,district,driver,d):
    try:
        alert = driver.switch_to.alert
        alert.dismiss()
        print ("alert accepted")
    except:
        pass
    
    e1=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_F_year'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_proceed").click()
    
    h2 = driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_dist')
    s_e2=Select(h2)
    s_e2.select_by_value(block)
    
    h21=(driver.find_element(By.NAME,'ctl00$ContentPlaceHolder1$ddl_block'))
    s_e21=Select(h21)
    s_e21.select_by_value(gram)
    
    h3=driver.find_element(By.NAME, 'ctl00$ContentPlaceHolder1$ddl_gpanchayat')
    s_e3=Select(h3)
    s_e3.select_by_value(gp)
    
    
    
    #
    try:
        try:
            driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_submit").click()
            
        except:
            element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_btn_submit")))
            driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_submit").click()
            
        if UnexpectedAlertPresentException:
            
            try:
                element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_grd_ShowData")))
                df=(pd.read_html(driver.find_element(By.ID,'ctl00_ContentPlaceHolder1_grd_ShowData').get_attribute('outerHTML'))[0])
                df['District']=(driver.find_element(By.ID,'ctl00_ContentPlaceHolder1_lbl_dist').get_attribute('outerHTML').split('>')[1].split('<')[0])
                df['Block']=(driver.find_element(By.ID,'ctl00_ContentPlaceHolder1_lbl_block_disp').get_attribute('outerHTML').split('>')[1].split('<')[0])
                df['GP']=(driver.find_element(By.ID,'ctl00_ContentPlaceHolder1_lbl_gp').get_attribute('outerHTML').split('>')[1].split('<')[0])
                data_append(df)
                element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "ctl00_ContentPlaceHolder1_btn_cancel")))
                driver.find_element(By.ID,"ctl00_ContentPlaceHolder1_btn_cancel").click()
                print('data entry started')
            finally:
                print('Pass called')
                pass
            print('data entry started')
    except UnexpectedAlertPresentException:
        print('Except loop')
        try:
            alert = driver.switch_to.alert
            alert.dismiss()
        except:
            pass
        


In [ ]:
# Uses Selenium Manager (Selenium >= 4.6) to locate ChromeDriver automatically.
# To use a specific driver instead, set the CHROMEDRIVER_PATH environment variable.
CHROMEDRIVER_PATH = os.environ.get("CHROMEDRIVER_PATH")
service = Service(CHROMEDRIVER_PATH) if CHROMEDRIVER_PATH else Service()
driver = webdriver.Chrome(service=service, options=options)
driver.get(url)

In [ ]:
for d in range(2022,2023):
    yr=str(int(d))
    data_df=pd.DataFrame()
    h2_driver(yr,driver,yr)
    data_df.to_excel(os.path.join(base_loc,yr+'.xlsx'))

In [ ]:
data_df.to_excel(os.path.join(base_loc,yr+'_5.xlsx'))

In [ ]:
data_df